In [0]:
dbutils.widgets.text("source_system", "", "source system")
dbutils.widgets.text("table_name", "", "table Name")
dbutils.widgets.text("landing_timestamp", "", "landing timestamp")
dbutils.widgets.text("source_file_format", "", "source file format")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
landing_path = f"/Volumes/azuredatabricks_medical/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

try:
    files = dbutils.fs.ls(landing_path)
    if len(files) == 0:
        print("No new files are received.")
        dbutils.notebook.exit("No new files received")
except Exception:
    print("No new files are received.")
    dbutils.notebook.exit("No new files received")

print(f"Files found at: {landing_path}")

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

In [0]:
from pyspark.sql import functions as F

updated_df = landing_df.withColumn("landing_timestamp", F.lit(landing_timestamp))\
                        .withColumn("current_timestamp", F.current_timestamp())

In [0]:
updated_df.write.mode("append").saveAsTable(f"azuredatabricks_medical.bronze.{table_name}")